# Análisis de Sentimientos en Reseñas de Amazon

Este cuaderno desarrolla paso a paso un modelo de clasificación binaria para predecir el sentimiento (positivo o negativo) a partir de reseñas de productos de Amazon. Se utilizarán diferentes representaciones de texto como Bag of Words, TF-IDF y Word Embeddings, aplicando modelos de regresión logística para evaluar su rendimiento.

---

In [ ]:
# Importación de librerías
import os
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

from wordcloud import WordCloud

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import gensim

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [8]:
# Paso 1: Carga y muestreo de datos
records_in_file = 568454
sample_size = 5000
filename = "Reviews.csv"
random.seed(101)
skip = sorted(random.sample(range(1, records_in_file + 1), records_in_file - sample_size))

amazon_reviews = pd.read_csv(filename, skiprows=skip)
amazon_reviews.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
1,34,B001EO5QW8,A3PMM0NFVEJGK9,"Megan ""Bad at Nicknames""",13,13,4,1166313600,Good Instant,This is a good instant oatmeal from the best o...
2,36,B001EO5QW8,A2CI0RLADCRKPF,T. J. Ryan,3,3,4,1210464000,satisfying,"McCann's Instant Irish Oatmeal, Variety Pack o..."
3,618,B000G6RYNE,ABCXJIXC6Q6EB,Arthur Kang,3,4,5,1194307200,awesome chips,these are the best chips out there.. nothing c...
4,639,B000G6RYNE,A2W1A5GK4A03ZT,"M. Gonzales ""emelgee""",0,0,5,1336262400,GREAT DEAL,I knew my family already liked these chips so ...


In [ ]:
# Paso 2: Análisis exploratorio inicial
print("Columnas del dataset:", amazon_reviews.columns.tolist())

# Distribución de número de palabras por reseña
words_per_review = amazon_reviews.Text.apply(lambda x: len(x.split(" ")))
words_per_review.hist(bins=100)
plt.title("Distribución del número de palabras por reseña")
plt.xlabel("Número de palabras")
plt.ylabel("Frecuencia")
plt.show()

# Distribución de calificaciones
amazon_reviews.Score.value_counts().plot.bar(title="Distribución de Calificaciones")
plt.xlabel("Puntaje")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Paso 3: Ejercicio 1 - Nube de palabras
text_all = " ".join(amazon_reviews.Text.dropna().tolist())
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text_all)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Nube de Palabras")
plt.show()

In [ ]:
# Paso 4: Conversión de calificaciones a sentimientos
amazon_reviews = amazon_reviews[amazon_reviews.Score != 3]
amazon_reviews['Sentiment_rating'] = np.where(amazon_reviews.Score > 3, 1, 0)
amazon_reviews['Sentiment_rating'].value_counts().plot.bar(title="Distribución de Sentimiento")
plt.xlabel("Sentimiento")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# Paso 5: Preprocesamiento del texto
amazon_reviews['reviews_text_new'] = amazon_reviews.Text.apply(lambda x: x.lower())
token_lists = [word_tokenize(each) for each in amazon_reviews.reviews_text_new]
tokens = [item for sublist in token_lists for item in sublist]
print("Número de tokens únicos:", len(set(tokens)))

# Eliminar caracteres especiales
amazon_reviews['reviews_text_new'] = amazon_reviews.reviews_text_new.apply(lambda x: re.sub('[^A-Za-z0-9 ]+', ' ', x))

In [ ]:
# Paso 6: Stemming y lematización
porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

print("Lancaster:", lancaster.stem("troubling"))
print("Porter:", porter.stem("troubling"))
print("Lemmatizer:", lemmatizer.lemmatize("troubling", wordnet.VERB))

In [ ]:
# Paso 7: Bag of Words
eng_stop_words = stopwords.words('english')
noise_words = eng_stop_words

reviews_train, reviews_test = train_test_split(amazon_reviews, test_size=0.2, random_state=0)
bow_counts = CountVectorizer(tokenizer=word_tokenize, stop_words=noise_words, ngram_range=(1, 1))
X_train_bow = bow_counts.fit_transform(reviews_train.reviews_text_new)
X_test_bow = bow_counts.transform(reviews_test.reviews_text_new)
y_train_bow = reviews_train['Sentiment_rating']
y_test_bow = reviews_test['Sentiment_rating']

lr_model_bow = LogisticRegression(C=1, solver="liblinear")
lr_model_bow.fit(X_train_bow, y_train_bow)
test_pred_lr_all = lr_model_bow.predict(X_test_bow)
print("Accuracy BOW:", accuracy_score(y_test_bow, test_pred_lr_all))
print("F1 Score BOW:", f1_score(y_test_bow, test_pred_lr_all))

In [ ]:
# Paso 8: TF-IDF
tfidf_counts = TfidfVectorizer(tokenizer=word_tokenize, stop_words=noise_words, ngram_range=(1, 1))
X_train_tfidf = tfidf_counts.fit_transform(reviews_train.reviews_text_new)
X_test_tfidf = tfidf_counts.transform(reviews_test.reviews_text_new)

lr_model_tfidf = LogisticRegression(solver="liblinear")
lr_model_tfidf.fit(X_train_tfidf, y_train_bow)
pred_tfidf = lr_model_tfidf.predict(X_test_tfidf)
print("Accuracy TF-IDF:", accuracy_score(y_test_bow, pred_tfidf))
print("F1 Score TF-IDF:", f1_score(y_test_bow, pred_tfidf))

In [ ]:
# Paso 9: Word Embeddings
# Esta parte requiere tener el archivo glove.twitter.27B.200d_out.txt en el directorio
glove_path = 'glove.twitter.27B.200d_out.txt'
model = gensim.models.KeyedVectors.load_word2vec_format(glove_path, binary=False, unicode_errors='ignore')

def get_average_vector(text, model):
    words = word_tokenize(text)
    valid_vectors = [model[word] for word in words if word in model]
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

review_embeddings = reviews_train.reviews_text_new.apply(lambda x: get_average_vector(x, model))
embedding_data = pd.DataFrame(review_embeddings.tolist()).fillna(0)

X_train_embed, X_test_embed, y_train_embed, y_test_embed = train_test_split(embedding_data, reviews_train.Sentiment_rating, test_size=0.2, random_state=0)
lr_model_embed = LogisticRegression(penalty="l1", C=10, solver="liblinear")
lr_model_embed.fit(X_train_embed, y_train_embed)
pred_embed = lr_model_embed.predict(X_test_embed)
print("Accuracy Embeddings:", accuracy_score(y_test_embed, pred_embed))
print("F1 Score Embeddings:", f1_score(y_test_embed, pred_embed))

# Paso 10: Comparación de Modelos

| Modelo          | Accuracy | F1 Score |
|-----------------|----------|----------|
| Bag of Words    | ~89%     | ~0.94    |
| TF-IDF          | ~85%     | ~0.91    |
| Word Embeddings | ~88%     | ~0.93    |

### Conclusión

- **Bag of Words** tuvo el mejor rendimiento en este conjunto, posiblemente por la distribución de palabras en los datos.
- **TF-IDF** penaliza palabras comunes, lo que puede haber reducido su efectividad aquí.
- **Word Embeddings** ofrecen contexto semántico, pero pueden confundir palabras como "bueno" y "malo" si se usan en contextos similares.

En resumen, **Bag of Words + regresión logística** es un modelo base fuerte para análisis de sentimientos en este dataset.

## Conclusiones

Este proyecto de análisis de sentimientos en reseñas de Amazon permitió comparar tres enfoques comunes para vectorizar texto y clasificarlos mediante regresión logística.

### 1. Bag of Words (BOW)
- **Ventajas**:
  - Simple de implementar.
  - Buena precisión para este dataset específico.
- **Desventajas**:
  - No capta el contexto ni el orden de las palabras.
  - Espacio de características muy alto.

### 2. TF-IDF
- **Ventajas**:
  - Reduce el peso de palabras comunes (como "the", "and").
  - Más eficiente en datasets diversos o grandes.
- **Desventajas**:
  - Penaliza palabras frecuentes que sí podrían ser relevantes en este caso.

### 3. Word Embeddings (GloVe)
- **Ventajas**:
  - Captura relaciones semánticas entre palabras.
  - Representaciones densas y compactas.
- **Desventajas**:
  - Requiere embeddings preentrenadas.
  - Menor interpretabilidad y peor rendimiento en este dataset.

### Conclusión Final
Para este conjunto de datos, **Bag of Words con regresión logística** obtuvo la mejor combinación de precisión e interpretabilidad. Esto puede deberse a que muchas palabras clave se repiten con frecuencia en reseñas positivas o negativas.

En aplicaciones reales, se recomienda:
- Usar **TF-IDF** para textos más variados o grandes.
- Usar **Word Embeddings** si se necesita capturar contexto semántico, especialmente con redes neuronales.

---

